In [1]:
# 3.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
import nltk
import warnings
from nltk.sentiment.vader import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# Konfigurasi path
DATA_PATH = '../../outputs/VTB/data_translated.csv'
OUTPUT_DIR = '../../outputs/VTB'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Memastikan Lexicon VADER tersedia
try:
    nltk.data.find('sentiment/vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon')

print("[INFO] Library dan Lexicon VADER berhasil dimuat.")

[INFO] Library dan Lexicon VADER berhasil dimuat.


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [2]:
# 3.2 Load Data dan Mengsi Data Kosong 
df = pd.read_csv(DATA_PATH)
df_clean = df.copy()

# 1. Isi NaN dengan string kosong
df_clean['teks_translated'] = df_clean['teks_translated'].fillna('')

# 2. Identifikasi baris yang kosong setelah di-strip
mask_kosong = df_clean['teks_translated'].str.strip() == ''

# 3. Isi dengan placeholder netral
df_clean.loc[mask_kosong, 'teks_translated'] = 'neutral'

print(f"Total data setelah pengisian: {len(df_clean)} tweet (konsisten dengan pendekatan lain)")
print(f"Jumlah baris yang diisi placeholder: {mask_kosong.sum()}")

Total data setelah pengisian: 450 tweet (konsisten dengan pendekatan lain)
Jumlah baris yang diisi placeholder: 0


In [3]:
# 3.3 Inisialisasi Analyzer
# Inisialisasi Analyzer dari library asli
analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    """Mengambil dictionary skor lengkap (neg, neu, pos, compound)"""
    return analyzer.polarity_scores(text)

print("\n[PROSES] Menghitung skor sentimen menggunakan VADER Original...")

# Terapkan scoring ke kolom teks_translated
scores_list = df_clean['teks_translated'].apply(get_vader_scores)

# Ekstraksi skor
df_clean['neg'] = scores_list.apply(lambda x: x['neg'])
df_clean['neu'] = scores_list.apply(lambda x: x['neu'])
df_clean['pos'] = scores_list.apply(lambda x: x['pos'])
df_clean['compound_score'] = scores_list.apply(lambda x: x['compound'])

print("[INFO] Perhitungan skor selesai.")


[PROSES] Menghitung skor sentimen menggunakan VADER Original...
[INFO] Perhitungan skor selesai.


In [4]:
# 3.3 Klasifikasi Sentimen berdasarkan Threshold VADER
def classify_vader(compound):
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

print("\n[PROSES] Klasifikasi sentimen berdasarkan threshold standar...")
df_clean['sentiment'] = df_clean['compound_score'].apply(classify_vader)

# Tampilkan Distribusi
distribusi = df_clean['sentiment'].value_counts()
persentase = df_clean['sentiment'].value_counts(normalize=True) * 100

print("\n=== DISTRIBUSI SENTIMEN VTB ===")
for label, count in distribusi.items():
    print(f"{label.upper():<10}: {count:>5} tweet ({persentase[label]:.2f}%)")


[PROSES] Klasifikasi sentimen berdasarkan threshold standar...

=== DISTRIBUSI SENTIMEN VTB ===
POSITIVE  :   204 tweet (45.33%)
NEUTRAL   :   130 tweet (28.89%)
NEGATIVE  :   116 tweet (25.78%)


In [5]:
# 3.4 Preview Hasil
print("\n[PREVIEW] 5 Data Pertama:")
preview_cols = ['no', 'teks_processed', 'teks_translated', 'compound_score', 'sentiment']
print(df_clean[preview_cols].head())


[PREVIEW] 5 Data Pertama:
    no                                     teks_processed  \
0   50  komisi IX ketua komisi IX DPR RI rekomendasi k...   
1   84  dapilku rumah reses dan silaturahim sama anggo...   
2   89  haru wakil rakyat terlalu banyak waktu untuk u...   
3  101  ketua dan para wakil ketua DPR RI PERLU RUU LA...   
4  121  viva anggota DPR minta TNI AU investigasi cela...   

                                     teks_translated  compound_score sentiment  
0  Commission IX Chairman of Commission IX DPR RI...          0.2263  positive  
1  my electoral district, home, constituency, out...          0.6249  positive  
2  haru representatives legislators have too much...         -0.2280  negative  
3  The chairman and deputy chairman of the DPR RI...          0.0000   neutral  
4  viva DPR members ask the Indonesian Air Force ...         -0.0772  negative  


In [6]:
# 3.5 Simpan Hasil Analisis
# Pilih kolom yang relevan untuk file final
final_cols = [
    'no', 'timestamp', 'teks', 'teks_processed', 
    'teks_translated', 'neg', 'neu', 'pos', 
    'compound_score', 'sentiment'
]

OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sentiment_vtb.csv')
df_clean[final_cols].to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[OUTPUT] Hasil analisis sentimen VTB disimpan di: {OUTPUT_FILE}")


[OUTPUT] Hasil analisis sentimen VTB disimpan di: ../../outputs/VTB\sentiment_vtb.csv
